In [1]:
!pip install scikit-learn pandas numpy matplotlib seaborn optuna
!pip install xgboost lightgbm catboost


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.4/364.4 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.5/233.5 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.6/78.6 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.7/98.7 MB 8.3 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, mean_squared_error, average_precision_score
from sklearn.metrics import ndcg_score

from catboost import CatBoostRanker, Pool

import optuna
import warnings
warnings.filterwarnings("ignore")


In [3]:
n_users = 2000
n_tracks = 1000

np.random.seed(42)
user_ids = np.arange(n_users)
user_ages = np.random.randint(15, 60, size=n_users)
user_genders = np.random.choice(['M', 'F'], size=n_users, p=[0.5, 0.5])
preferred_genres = np.random.choice(['rock', 'pop', 'rap', 'jazz'], size=n_users)

users_df = pd.DataFrame({
    'user_id': user_ids,
    'age': user_ages,
    'gender': user_genders,
    'preferred_genre': preferred_genres
})

track_ids = np.arange(n_tracks)
track_genres = np.random.choice(['rock', 'pop', 'rap', 'jazz'], size=n_tracks)
artist_ids = np.random.randint(1, 300, size=n_tracks)
tracks_df = pd.DataFrame({
    'track_id': track_ids,
    'genre': track_genres,
    'artist_id': artist_ids
})

# Предположим, что каждый пользователь взаимодействует примерно с 10% доступных треков
interaction_data = []
for user_id in user_ids:
    # Случайное подмножество треков для прослушивания
    interacted_tracks = np.random.choice(track_ids, size=int(n_tracks*0.1), replace=False)
    for t_id in interacted_tracks:
        # "Лайк" - 0 или 1
        like = np.random.choice([0, 1], p=[0.7, 0.3])
        # "Дослушал до конца" - 0 или 1
        listened_full = np.random.choice([0, 1], p=[0.4, 0.6])
        # "Добавил в плейлист" - 0 или 1
        playlist_add = np.random.choice([0, 1], p=[0.85, 0.15])

        interaction_data.append([user_id, t_id, like, listened_full, playlist_add])

interactions_df = pd.DataFrame(interaction_data,
                               columns=['user_id','track_id','like','listened_full','playlist_add'])

print("users_df shape:", users_df.shape)
print("tracks_df shape:", tracks_df.shape)
print("interactions_df shape:", interactions_df.shape)

users_df.head()


users_df shape: (2000, 4)
tracks_df shape: (1000, 3)
interactions_df shape: (200000, 5)


,user_id,age,gender,preferred_genre
0,0,53,F,pop
1,1,43,M,jazz
2,2,29,F,pop
3,3,57,F,rock
4,4,22,M,pop


In [4]:
merged_df = interactions_df.merge(users_df, on='user_id', how='left')

merged_df = merged_df.merge(tracks_df, on='track_id', how='left')

merged_df.head()


,user_id,track_id,like,listened_full,playlist_add,age,gender,preferred_genre,genre,artist_id
0,0,443,1,1,0,53,F,pop,pop,238
1,0,291,0,1,0,53,F,pop,rap,244
2,0,156,1,1,0,53,F,pop,jazz,44
3,0,488,0,0,0,53,F,pop,rock,81
4,0,159,0,0,1,53,F,pop,rap,242


In [5]:
print("Missing values:\n", merged_df.isna().sum())


Missing values:
 user_id            0
track_id           0
like               0
listened_full      0
playlist_add       0
age                0
gender             0
preferred_genre    0
genre              0
artist_id          0
dtype: int64


In [6]:
merged_df['target'] = (5*merged_df['like']
                       + 3*merged_df['listened_full']
                       + 8*merged_df['playlist_add'])


In [9]:
!pip install --upgrade scikit-learn
cat_cols = ['gender', 'preferred_genre', 'genre']
encoder = OneHotEncoder(sparse_output=False)

encoded = encoder.fit_transform(merged_df[cat_cols])
encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(cat_cols))

numerical_cols = ['age', 'artist_id']
scaler = StandardScaler()
scaled = scaler.fit_transform(merged_df[numerical_cols])
scaled_df = pd.DataFrame(scaled, columns=[f"{col}_scaled" for col in numerical_cols])

features_df = pd.concat(
    [merged_df[['user_id','track_id']], encoded_df, scaled_df],
    axis=1
)

features_df['target'] = merged_df['target'].values
features_df.head()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.5/13.5 MB 78.3 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.0
    Uninstalling scikit-learn-1.6.0:
      Successfully uninstalled scikit-learn-1.6.0


,user_id,track_id,gender_F,gender_M,preferred_genre_jazz,preferred_genre_pop,preferred_genre_rap,preferred_genre_rock,genre_jazz,genre_pop,genre_rap,genre_rock,age_scaled,artist_id_scaled,target
0,0,443,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.21391,1.072758,8
1,0,291,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.21391,1.142290,3
2,0,156,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.21391,-1.175441,8
3,0,488,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.21391,-0.746660,0
4,0,159,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.21391,1.119113,8


In [10]:
train_users, test_users = train_test_split(users_df['user_id'], test_size=0.2, random_state=42)

train_data = features_df[features_df['user_id'].isin(train_users)].copy()
test_data = features_df[features_df['user_id'].isin(test_users)].copy()

print("Train shape:", train_data.shape, "Test shape:", test_data.shape)


Train shape: (160000, 15) Test shape: (40000, 15)


In [11]:
def get_group_sizes(df):
    return df.groupby('user_id').size().values

train_groups = get_group_sizes(train_data)
test_groups = get_group_sizes(test_data)

train_pool = Pool(
    data=train_data.drop(['user_id','track_id','target'], axis=1),
    label=train_data['target'],
    group_id=train_data['user_id'],
    group_weight=None,
    cat_features=[]
)

test_pool = Pool(
    data=test_data.drop(['user_id','track_id','target'], axis=1),
    label=test_data['target'],
    group_id=test_data['user_id'],
    group_weight=None,
    cat_features=[]
)


In [12]:
model_ranker = CatBoostRanker(
    iterations=100,
    learning_rate=0.1,
    random_seed=42,
    verbose=50
)

model_ranker.fit(train_pool, eval_set=test_pool)

0:	test: 0.6934758	best: 0.6934758 (0)	total: 840ms	remaining: 1m 23s
50:	test: 0.7407677	best: 0.7414440 (48)	total: 29.3s	remaining: 28.2s
99:	test: 0.7411981	best: 0.7419695 (85)	total: 59.1s	remaining: 0us

bestTest = 0.7419694983
bestIteration = 85

Shrink model to first 86 iterations.


In [13]:
test_data['pred'] = model_ranker.predict(test_data.drop(['user_id','track_id','target'], axis=1))

ndcg_list = []
for user_id, group in test_data.groupby('user_id'):
    true_scores = group['target'].values
    preds = group['pred'].values

    ndcg_val = ndcg_score([true_scores], [preds], k=5)
    ndcg_list.append(ndcg_val)

avg_ndcg_5 = np.mean(ndcg_list)
print(f"Средний NDCG@5 по всем пользователям: {avg_ndcg_5:.4f}")


Средний NDCG@5 по всем пользователям: 0.3142


In [14]:
some_user = test_data['user_id'].iloc[0]

user_df = test_data[test_data['user_id'] == some_user].copy()

user_df['pred_score'] = model_ranker.predict(user_df.drop(['user_id','track_id','target','pred'], axis=1))

recommended_tracks = user_df.sort_values('pred_score', ascending=False)
recommended_tracks = recommended_tracks.head(10)

print("Топ-10 треков для пользователя", some_user)
recommended_tracks[['track_id','pred_score','target']].head(10)


Топ-10 треков для пользователя 23


,track_id,pred_score,target
2385,617,0.119627,3
2322,65,0.107264,3
2336,890,0.096443,3
2374,100,0.096367,0
2395,876,0.087848,3
2305,638,0.072650,3
2361,733,0.072451,3
2372,560,0.059203,5
2303,921,0.056057,8
2314,135,0.046412,3
